In [ ]:
import torch
import torch.nn as nn
import timm
import torchvision.transforms as transforms
from PIL import Image
from flask import Flask, request, jsonify
import io
import base64
import geopandas as gpd
from shapely.geometry import Polygon, Point
import json
import random

app = Flask(__name__)
# fmt: off
country_names = ["Albania", "Andorra", "Antarctica", "Argentina", "Australia", "Austria", "Bangladesh", "Belgium", "United Kingdom", "Bhutan", "Bolivia", "Botswana", "Brazil", "Bulgaria", "Cambodia", "Canada", "Chile", "Australia", "Australia", "Colombia", "Costa Rica", "Croatia", "Netherlands", "Czechia", "Denmark", "Dominican Republic", "Ecuador", "Estonia", "Swaziland", "Denmark", "Finland", "France", "Germany", "Ghana", "United Kingdom", "Greece", "Denmark", "United States", "Guatemala", "China", "Hungary", "Iceland", "India", "Ireland", "United Kingdom", "Israel", "Italy", "Japan", "United Kingdom", "Jordan", "Kazakhstan", "Kenya", "Kyrgyzstan", "Laos", "Latvia", "Lebanon", "Lesotho", "Liechtenstein", "Lithuania", "Luxembourg", "Malta", "Mexico", "Monaco", "Mongolia", "Montenegro", "Namibia", "Nepal", "Netherlands", "New Zealand", "Nigeria", "Macedonia", "United States", "Norway", "Oman", "Pakistan", "Panama", "Peru", "Philippines", "Poland", "Portugal", "United States", "Qatar", "Romania", "Rwanda", "France", "San Marino", "Senegal", "Serbia", "Singapore", "Slovakia", "Slovenia", "South Africa", "Korea, South", "Spain", "Sri Lanka", "Sweden", "Switzerland", "Taiwan", "Thailand", "Tunisia", "Turkey", "United States", "Uganda", "Ukraine", "United Arab Emirates", "United States", "Vietnam", "Indonesia", "Malaysia", "Russia", "United Kingdom", "Uruguay",
]
# fmt: on
COUNTRY_MODEL_PATH = "tinyvit_country.pth"
SQUARE_MODEL_PATH = "tinyvit_squares.pth"
NUM_COUNTRY_CLASSES = len(country_names)
NUM_SQUARE_CLASSES = 3855
gdf = gpd.read_file("countryBoundaries.geojson")

with open("label_mapping.json", "r") as f:
    squareLabels = json.load(f)

finalCoords = {"lng": 0.0, "lat": 0.0}


# CORS headers for all responses
@app.after_request
def after_request(response):
    response.headers.add("Access-Control-Allow-Origin", "*")
    response.headers.add("Access-Control-Allow-Headers", "Content-Type,Authorization")
    response.headers.add("Access-Control-Allow-Methods", "GET,PUT,POST,DELETE,OPTIONS")
    return response


# model architecture for country
class CountryCustomTinyViT(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        in_features = backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x


# model architecture for square
class SquareCustomTinyViT(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        in_features = backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x


try:
    print(f"Loading model...")

    backbone_country = timm.create_model(
        "tiny_vit_21m_224", pretrained=True, num_classes=0
    )
    backbone_square = timm.create_model(
        "tiny_vit_11m_224", pretrained=True, num_classes=0
    )

    countryModel = CountryCustomTinyViT(backbone_country, NUM_COUNTRY_CLASSES)
    squareModel = SquareCustomTinyViT(backbone_square, NUM_SQUARE_CLASSES)

    # Load the state dictionary
    countryModelState = torch.load(COUNTRY_MODEL_PATH, map_location="cpu")
    squareModelState = torch.load(
        SQUARE_MODEL_PATH, map_location="cpu"
    )
    squareModelState.keys()
    print(squareModel.classifier)

    # Check if the state dict matches the model architecture
    try:
        countryModel.load_state_dict(countryModelState)
        squareModel.load_state_dict(squareModelState)

    except RuntimeError as e:
        print(f"Error loading state dict: {e}")
        raise

    countryModel.eval()
    squareModel.eval()
    print(f"Models loaded successfully")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

# match training transformations
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


def is_square_in_country(square_id, country_id):

    try:
        country = country_names[country_id]
        # print(f"  Looking for country: {country}")

        country_row = gdf[gdf["shapeName"] == country]

        # Get the first geometry (assuming one country per name)
        country_geometry = country_row.geometry.iloc[0]
        # print(f"  Country geometry type: {type(country_geometry)}")

        # all coord labels are in style "30.0_40.0_50.0_60.0" for min_lat,max_lat,min_lon,max_lon
        square_coords_str = squareLabels.get(str(square_id))
        if not square_coords_str:
            print(f"  Square ID {square_id} not found in mapping")
            return False

        square_coord_array = square_coords_str.split("_")

        min_lat, max_lat, min_lon, max_lon = map(float, square_coord_array)
        # print(f"  Square bounds: ({min_lat}, {max_lat}) to ({min_lon}, {max_lon})")

        square_coords = [
            (min_lon, max_lat),  # top-left
            (max_lon, max_lat),  # top-right
            (max_lon, min_lat),  # bottom-right
            (min_lon, min_lat),  # bottom-left
            (min_lon, max_lat),  # close the polygon
        ]

        square_polygon = Polygon(square_coords)

        # Check if the square intersects with the country
        intersects = country_geometry.intersects(square_polygon)
        if intersects:
            global finalCoords
            # Update final coordinates to the center of the square
            finalCoords["lng"] = (min_lon + max_lon) / 2
            finalCoords["lat"] = (min_lat + max_lat) / 2
            print(
                f" Min lat: {min_lat}, Max lat: {max_lat}, Min lon: {min_lon}, Max lon: {max_lon}"
            )
            finalCoords["lat"], finalCoords["lng"] = is_coords_in_country(
                finalCoords["lat"],
                finalCoords["lng"],
                country_geometry,
                min_lat,
                max_lat,
                min_lon,
                max_lon,
                0,  # initial attempt
            )

            print(f"  Updated final coordinates: {finalCoords}")

        return intersects

    except Exception as e:
        print(f"  Error in is_square_in_country: {e}")
        import traceback

        traceback.print_exc()
        return False


def is_coords_in_country(
    lat, lng, country_geometry, min_lat, max_lat, min_lon, max_lon, attempts
):
#Checks if point and country are overlapping
    if attempts >= 490:
        print("Max attempts reached, returning random coordinates")
        return lat, lng
    point = Point(lng, lat)
    if country_geometry.contains(point):
        return lat, lng
    else:
        new_lat = random.uniform(min_lat, max_lat)
        new_lng = random.uniform(min_lon, max_lon)
        return is_coords_in_country(
            new_lat, new_lng, country_geometry, min_lat, max_lat, min_lon, max_lon, attempts + 1
        )


def prediction(img_path):

    image = Image.open(img_path).convert("RGB")

    input_tensor = transform(image).unsqueeze(0)
    with torch.no_grad():

        country_output = countryModel(input_tensor)
        square_output = squareModel(input_tensor)

        # single country prediction
        country_prediction = torch.argmax(country_output, dim=1).item()

        # many square predictions sorted by confidence
        square_probs = torch.softmax(square_output, dim=1)
        sorted_square_indices = torch.argsort(
            square_probs, dim=1, descending=True
        ).squeeze(0)

        print(
            f"Country prediction: {country_prediction} ({country_names[country_prediction]})"
        )
        print(f"Top 5 square predictions: {sorted_square_indices[:5].tolist()}")

        # Checks for highest confidence square prediction that is valid
        valid_square_prediction = None
        checked_squares = 0

        for square_idx in sorted_square_indices[:NUM_SQUARE_CLASSES]:
            square_candidate = square_idx.item()
            checked_squares += 1

            if is_square_in_country(square_candidate, country_prediction):
                valid_square_prediction = square_candidate
                print(
                    f"Found valid square: {square_candidate} after checking {checked_squares} squares"
                )
                break

    return finalCoords["lat"], finalCoords["lng"]





Loading model...
Sequential(
  (0): Linear(in_features=448, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=512, out_features=256, bias=True)
  (4): GELU(approximate='none')
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=256, out_features=3855, bias=True)
)
Models loaded successfully


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/dev/GeoguessrAI/testSetImageStitched.zip'
extract_path = '/content/data/images'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipped to:", extract_path)

zip_path = '/content/drive/MyDrive/dev/GeoguessrAI/testSetJson.zip'
extract_path = '/content/data/json'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipped to:", extract_path)

✅ Unzipped to: /content/data/images
✅ Unzipped to: /content/data/json


In [ ]:
CLASS_NAMES = {
    0: "Albania", 1: "Andorra", 2: "Antarctica", 3: "Argentina",
    4: "Australia", 5: "Austria", 6: "Bangladesh", 7: "Belgium",
    8: "Bermuda", 9: "Bhutan", 10: "Bolivia", 11: "Botswana",
    12: "Brazil", 13: "Bulgaria", 14: "Cambodia", 15: "Canada",
    16: "Chile", 17: "Christmas Island", 18: "Cocos (Keeling) Islands",
    19: "Colombia", 20: "Costa Rica", 21: "Croatia", 22: "Curaçao",
    23: "Czechia", 24: "Denmark", 25: "Dominican Republic",
    26: "Ecuador", 27: "Estonia", 28: "Eswatini", 29: "Faroe Islands",
    30: "Finland", 31: "France", 32: "Germany", 33: "Ghana",
    34: "Gibraltar", 35: "Greece", 36: "Greenland", 37: "Guam",
    38: "Guatemala", 39: "Hong Kong", 40: "Hungary", 41: "Iceland",
    42: "India", 43: "Ireland", 44: "Isle of Man", 45: "Israel",
    46: "Italy", 47: "Japan", 48: "Jersey", 49: "Jordan",
    50: "Kazakhstan", 51: "Kenya", 52: "Kyrgyzstan", 53: "Laos",
    54: "Latvia", 55: "Lebanon", 56: "Lesotho", 57: "Liechtenstein",
    58: "Lithuania", 59: "Luxembourg", 60: "Malta", 61: "Mexico",
    62: "Monaco", 63: "Mongolia", 64: "Montenegro", 65: "Namibia",
    66: "Nepal", 67: "Netherlands", 68: "New Zealand", 69: "Nigeria",
    70: "North Macedonia", 71: "Northern Mariana Islands", 72: "Norway",
    73: "Oman", 74: "Pakistan", 75: "Panama", 76: "Peru",
    77: "Philippines", 78: "Poland", 79: "Portugal", 80: "Puerto Rico",
    81: "Qatar", 82: "Romania", 83: "Rwanda", 84: "Réunion",
    85: "San Marino", 86: "Senegal", 87: "Serbia", 88: "Singapore",
    89: "Slovakia", 90: "Slovenia", 91: "South Africa",
    92: "South Korea", 93: "Spain", 94: "Sri Lanka", 95: "Sweden",
    96: "Switzerland", 97: "Taiwan", 98: "Thailand", 99: "Tunisia",
    100: "Turkey", 101: "US Virgin Islands", 102: "Uganda",
    103: "Ukraine", 104: "United Arab Emirates", 105: "United States",
    106: "Vietnam", 107: "indo", 108: "malay", 109: "russia",
    110: "uk", 111: "uruguay"
}

In [ ]:
import math

#calculate distance in km between to coordinate point
def haversine_km(lat1, lon1, lat2, lon2, R=6371.0088):
    φ1, λ1, φ2, λ2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dφ = φ2 - φ1
    dλ = λ2 - λ1
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    c = 2*math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c


DATASET_DIR = "/content/data/images/testSetImageStitched"
JSON_DIR = "/content/data/json/testSetJson"
distanceJson={}

test_class_folders = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])
print(len(test_class_folders))
for true_class in test_class_folders:
    folder_path = os.path.join(DATASET_DIR, true_class)
    json_path = os.path.join(JSON_DIR, true_class + ".json")
    with open(json_path, 'r') as f:
        json_data = json.load(f)
    json_data=json_data["customCoordinates"]

    for fname in os.listdir(folder_path):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        imgMatch = next((item for item in json_data if item["panoId"]==fname.split(" ")[0]))
        img_path = os.path.join(folder_path, fname)

        pred_lat, pred_lng=prediction(img_path)
        print(f"Predicted coordinates: {pred_lat}, {pred_lng}")
        print(f"Real Coords: {imgMatch['lat'], imgMatch['lng']}")
        distance = haversine_km(imgMatch['lat'], imgMatch['lng'], pred_lat, pred_lng)
        if true_class not in distanceJson:
          distanceJson[true_class] = []
        distanceJson[true_class].append(distance)
        print(f"Distance: {int(distance)}")


with open("distanceJson.json", "w") as f:
    json.dump(distanceJson, f)



Streaming output truncated to the last 5000 lines.
  Updated final coordinates: {'lng': -85.78125, 'lat': 37.265625}
Found valid square: 1912 after checking 1 squares
Predicted coordinates: 37.265625, -85.78125
Real Coords: (37.95154255947529, -85.05223944241853)
Distance: 99
Country prediction: 105 (United States)
Top 5 square predictions: [1647, 1806, 1665, 1654, 1742]
 Min lat: 28.125, Max lat: 29.53125, Min lon: -84.375, Max lon: -78.75
  Updated final coordinates: {'lng': -81.5625, 'lat': 28.828125}
Found valid square: 1647 after checking 1 squares
Predicted coordinates: 28.828125, -81.5625
Real Coords: (27.730007795315746, -82.01950272149638)
Distance: 130
Country prediction: 105 (United States)
Top 5 square predictions: [1888, 2185, 1912, 2058, 2280]
 Min lat: 36.5625, Max lat: 37.265625, Min lon: -90.0, Max lon: -87.1875
  Updated final coordinates: {'lng': -88.59375, 'lat': 36.9140625}
Found valid square: 1888 after checking 1 squares
Predicted coordinates: 36.9140625, -88.593

In [ ]:
from statistics import median
for key, value in distanceJson.items():
  print(f"{key}: {sum(value)/len(value)}")


for key, value in distanceJson.items():
  print(f"{key}: {median(value)}")


Albania (100 locations): 419.58293054038825
Andorra (100 locations): 376.90202660703
Argentina (100 locations): 600.7456526985203
Australia (100 locations): 478.4228728083918
Austria (100 locations): 555.2718156152454
Bangladesh (100 locations): 229.78179112287214
Belgium (100 locations): 463.3914093690943
Bermuda (100 locations): 3796.4824155625192
Bhutan (100 locations): 63.04773902856176
Bolivia (100 locations): 349.84583216131773
Botswana (100 locations): 460.3652425584832
Brazil (100 locations): 1222.1395227570765
Bulgaria (100 locations): 265.7668079611847
Cambodia (100 locations): 423.0005771514394
Canada (100 locations) (1): 842.7799933596996
Chile (100 locations): 788.538771328361
Christmas Island (100 locations): 1968.0197256806614
Cocos (Keeling) Islands (100 locations): 3827.7973755886924
Colombia (100 locations): 2013.1947200833768
Costa Rica (26 locations): 9700.410041376135
Croatia (100 locations): 605.7936735972296
Curaçao (100 locations): 188.99769333706976
Czechia (10

In [ ]:
total_sum = sum(sum(sublist) for sublist in distanceJson.values())
total_count = sum(len(sublist) for sublist in distanceJson.values())
average = total_sum / total_count
print(average)

total_median = median(median(sublist) for sublist in distanceJson.values())
print(total_median)

684.1467386635323
109.36617658089801
